# FinAgent Pro — Reproducible Experiment

This notebook re-runs a single analysis experiment end-to-end: pulls market data, trains LSTM models (if needed), computes expected returns, performs portfolio optimization, and backtests the resulting portfolio against two baselines: equal-weight and a simple moving-average (MA) filter baseline. It then produces equity curve and drawdown plots and a comparison table of metrics.

In [ ]:
# Setup imports and path
import os, sys
# Ensure repo root is on path (notebooks/ is under project root)
sys.path.insert(0, os.path.abspath('..'))
: 
,
: {
: 

: [
,
,
,
,
,
,
,
,
,
,
,

: 
,
: {
: 

: [
,
,
30
,
8
,
,

In [ ]:
# Run agents: market, sentiment, risk, prediction
market = run_market_data(TICKERS, period=HIST_PERIOD)
sentiment = run_sentiment(TICKERS)
risk = run_risk_analysis(TICKERS, period=HIST_PERIOD)
prediction = run_prediction(TICKERS, predict_days=PREDICT_DAYS, epochs=EPOCHS)
# Show a compact summary
expected_returns = {t: prediction.get(t, {}).get('expected_return', 0.0) for t in TICKERS}
print('Expected returns (annualized)')
for t, r in expected_returns.items():
    print(f'{t}: {r:.4f}')

In [ ]:
# Run portfolio optimization using the improved agent
allocation = run_portfolio_optimization(TICKERS, expected_returns, risk, risk_profile='moderate', history_period=HIST_PERIOD)
# Filter tiny weights
allocation = {k: v for k, v in allocation.items() if v >= 0.01}
print('Optimized allocation:')
print(allocation)
: 
,
: {
: 

: [
,
1
,
,
2
    df = pd.DataFrame()
    for t in tickers:
        hist = yf.download(t, period=period, progress=False)['Close'].rename(t)
        df = pd.concat([df, hist], axis=1)
    df = df.dropna()
    sma = df.rolling(window=sma_window).mean().iloc[-1]
    current = df.iloc[-1]
    selected = [t for t in tickers if current[t] > sma[t]]
    if len(selected) == 0:
        return {t: 1.0/len(tickers) for t in tickers}
    return {t: (1.0/len(selected) if t in selected else 0.0) for t in tickers}
ma_alloc = ma_filter_allocation(TICKERS, period=HIST_PERIOD)
print('Equal allocation:', equal_alloc)
print('MA-based allocation:', ma_alloc)
: 
,
: {
: 

: [
    df = pd.DataFrame()
    for t in tickers:
        s = yf.download(t, period=period, progress=False)['Close'].rename(t)
        df = pd.concat([df, s], axis=1)
    df = df.dropna()
    returns = df.pct_change().dropna()
    weights = pd.Series({t: allocation.get(t, 0) for t in returns.columns})
    weights = weights.reindex(returns.columns).fillna(0)
    if weights.sum() == 0:
        weights = pd.Series({t: 1.0/len(returns.columns) for t in returns.columns})
    weights = weights / weights.sum()
    port_daily = returns.dot(weights)
    cum = (1 + port_daily).cumprod()
    total_return = float(cum.iloc[-1] - 1)
    annual_return = float(port_daily.mean()*252)
    annual_vol = float(port_daily.std() * (252 ** 0.5))
    sharpe = float((annual_return - 0.05) / annual_vol) if annual_vol != 0 else None
    roll_max = cum.cummax()
    drawdown = (cum / roll_max) - 1
    max_dd = float(drawdown.min())
    metrics = {
        'daily_returns': port_daily,
        'annual_return': annual_return,
        'sharpe': sharpe,
    }
    return metrics
: 
,
: {
: 

: [
,
,
,
,
,
6
,
,
,
,
,
,
,
,
,
,
,
4
,
1
,
1
,
1
,
,
,
,
,
,

: 
,
: {
: 

: [
,
    {
        'total_return': opt_metrics['total_return'],
        'annual_volatility': opt_metrics['annual_volatility'],

# FinAgent Pro — Reproducible Experiment with Baselines

This notebook runs a reproducible experiment: pulls market data, trains LSTM models (if needed), computes expected returns, adds ARIMA and XGBoost baselines, performs portfolio optimization for each forecasting method, and backtests resulting portfolios. It produces equity curves, drawdowns, and a comparison table of metrics.

In [ ]:
# Setup imports and path
import os, sys
sys.path.insert(0, os.path.abspath('..'))
print('Repo root added to path')

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime
from agents.market_agent import run_market_data
from agents.sentiment_agent import run_sentiment
from agents.risk_agent import run_risk_analysis
from agents.prediction_agent import run_prediction
from agents.portfolio_agent import run_portfolio_optimization
# Baseline libs
from statsmodels.tsa.arima.model import ARIMA
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
print('Imports OK')

In [ ]:
# Experiment configuration
TICKERS = ['AAPL','MSFT','GOOGL','AMZN','NVDA']
PREDICT_DAYS = 30
EPOCHS = 20  # increased epochs for stronger training
HIST_PERIOD = '1y'
print('Tickers:', TICKERS)
: 
,
: {
: 
,
: 

: [
,
,
,
,
,
,

,
,

: 
,
: {
: 
,
: 

: [
,
,
,